# Hostile Testing Phase 2 (Corrected) - Config, Git, Admin

## Purpose
Fixed version with correct function signatures.

## Goal
Test config, git, and admin modules with proper parameters

In [1]:
import sys
import pandas as pd
import tempfile
from pathlib import Path
import shutil
import json

# Test results tracking
test_results = {
    'function': [],
    'module': [],
    'test_category': [],
    'passed': [],
    'error_message': [],
    'severity': []
}

def record_test(function, module, test_category, passed, error_message="", severity="medium"):
    """Record a test result."""
    test_results['function'].append(function)
    test_results['module'].append(module)
    test_results['test_category'].append(test_category)
    test_results['passed'].append(passed)
    test_results['error_message'].append(error_message)
    test_results['severity'].append(severity)

def hostile_test(func, test_name, *args, **kwargs):
    """Execute a hostile test and record results."""
    try:
        result = func(*args, **kwargs)
        return (True, result, "")
    except Exception as e:
        return (False, None, str(e))

print("Test harness loaded for Phase 2 (Corrected)")

Test harness loaded for Phase 2 (Corrected)


## Section 1: Config System - Database Configuration (Corrected)

In [2]:
from siege_utilities import (
    create_database_config, save_database_config, load_database_config,
    get_spark_database_options, list_database_configs
)

test_dir = Path(tempfile.mkdtemp())

try:
    # Hostile database config inputs (WITH CORRECT SIGNATURES)
    # create_database_config(name, connection_type, host, port, database, username, password, **kwargs)
    hostile_db_configs = [
        # SQL Injection in database names
        {"name": "'; DROP TABLE users; --", "connection_type": "postgres", "host": "localhost", 
         "port": 5432, "database": "testdb", "username": "user", "password": "pass"},
        # XSS in names
        {"name": "<script>alert('xss')</script>", "connection_type": "mysql", "host": "localhost",
         "port": 3306, "database": "testdb", "username": "user", "password": "pass"},
        # Path traversal in database param
        {"name": "test", "connection_type": "postgres", "host": "localhost",
         "port": 5432, "database": "../../../etc/passwd", "username": "user", "password": "pass"},
        # Very long strings
        {"name": "A" * 1000, "connection_type": "postgres", "host": "localhost",
         "port": 5432, "database": "B" * 1000, "username": "user", "password": "pass"},
        # Null bytes
        {"name": "test\x00db", "connection_type": "mysql", "host": "localhost\x00evil",
         "port": 3306, "database": "testdb", "username": "user", "password": "pass"},
        # Command injection in host
        {"name": "test", "connection_type": "postgres", "host": "localhost; rm -rf /",
         "port": 5432, "database": "testdb", "username": "user", "password": "pass"},
        # Unicode
        {"name": "数据库", "connection_type": "postgres", "host": "本地主机",
         "port": 5432, "database": "测试", "username": "用户", "password": "密码"},
        # URL-encoded
        {"name": "test%00admin", "connection_type": "mysql", "host": "localhost%2F..",
         "port": 3306, "database": "test", "username": "user", "password": "pass"},
    ]
    
    for i, config in enumerate(hostile_db_configs):
        # Test create_database_config
        success, result, error = hostile_test(
            create_database_config,
            f"hostile_db_config_{i}",
            **config
        )
        record_test(
            "create_database_config",
            "config.databases",
            "hostile_db_configs",
            success,
            error,
            severity="high" if "DROP TABLE" in str(config.get('name', '')) else "medium"
        )
        
        # If config was created, try to save it
        if success and result:
            success2, result2, error2 = hostile_test(
                save_database_config,
                f"save_hostile_db_{i}",
                result,
                str(test_dir)
            )
            record_test(
                "save_database_config",
                "config.databases",
                "save_hostile_configs",
                success2,
                error2,
                severity="medium"
            )
    
    # Test load_database_config with hostile inputs
    hostile_db_names = [
        "nonexistent",
        "../../../etc/passwd",
        "test; rm -rf /",
        "\x00null",
        "A" * 1000,
    ]
    
    for db_name in hostile_db_names:
        success, result, error = hostile_test(
            load_database_config,
            "load_hostile",
            db_name,
            str(test_dir)
        )
        record_test(
            "load_database_config",
            "config.databases",
            "hostile_db_names",
            success,
            error,
            severity="high" if "../" in str(db_name) else "medium"
        )
    
    print(f"Completed {len([r for r in test_results['module'] if r == 'config.databases'])} database config tests")
    
finally:
    shutil.rmtree(test_dir, ignore_errors=True)

Importing logging from siege_utilities.core.logging
Successfully imported 15 functions from logging
Importing string_utils from siege_utilities.core.string_utils
Successfully imported 1 functions from string_utils
siege_utilities.core: Imported 16 functions


[siege_utilities] 2025-10-13 08:37:34,252 INFO: Importing hdfs_operations from siege_utilities.distributed.hdfs_operations


[siege_utilities] 2025-10-13 08:37:34,253 INFO: Successfully imported 2 functions from hdfs_operations


[siege_utilities] 2025-10-13 08:37:34,253 INFO: Importing hdfs_config from siege_utilities.distributed.hdfs_config


[siege_utilities] 2025-10-13 08:37:34,255 INFO: Successfully imported 5 functions from hdfs_config


...

[siege_utilities] 2025-10-13 08:37:34,807 INFO: DuckDB not available - using standard geospatial stack


[siege_utilities] 2025-10-13 08:37:35,086 WARNING: Could not import additional analytics utilities: cannot import name 'get_dataset_metadata' from 'siege_utilities.analytics.datadotworld_connector' (/Users/dheerajchand/Documents/Professional/Siege_Analytics/Code/siege_utilities/siege_utilities/analytics/datadotworld_connector.py)


[siege_utilities] 2025-10-13 08:37:35,087 WARNING: Could not import reporting utilities: No module named 'siege_utilities.reporting.base_template'


[siege_utilities] 2025-10-13 08:37:35,087 WARNING: Could not import chart generation functions: No module named 'siege_utilities.reporting.base_template'


INFO: Importing environment from siege_utilities.testing.environment
INFO: Successfully imported 9 functions from environment
INFO: Importing runner from siege_utilities.testing.runner
INFO: Successfully imported 6 functions from runner
Created database config: '; DROP TABLE users; -- (postgres)
Saving database config with password in plain text to /var/folders/9h/83s_sxx17hv5gkccscgbyvzw0000gn/T/tmpfpxk4i0h/database_'; DROP TABLE users; --.json
In production, consider using environment variables or encryption
Saved database config to: /var/folders/9h/83s_sxx17hv5gkccscgbyvzw0000gn/T/tmpfpxk4i0h/database_'; DROP TABLE users; --.json
Created database config: <script>alert('xss')</script> (mysql)
Saving database config with password in plain text to /var/folders/9h/83s_sxx17hv5gkccscgbyvzw0000gn/T/tmpfpxk4i0h/database_<script>alert('xss')</script>.json
In production, consider using environment variables or encryption
Created database config: test (postgres)
Saving database config with pa

## Section 2: Config System - Project Configuration (Corrected)

In [3]:
from siege_utilities import (
    create_project_config, save_project_config, load_project_config,
    list_projects, update_project_config
)

test_dir = Path(tempfile.mkdtemp())

try:
    # Hostile project inputs (WITH CORRECT SIGNATURES)
    # create_project_config(project_name, project_root, description='', **kwargs)
    hostile_projects = [
        {"project_name": "../../../etc/passwd", "project_root": Path("/tmp/test")},
        {"project_name": "test; rm -rf /", "project_root": Path("/tmp/test")},
        {"project_name": "A" * 1000, "project_root": Path("/tmp/" + "B" * 100)},
        {"project_name": "test\x00project", "project_root": Path("/tmp/test")},
        {"project_name": "<script>alert('xss')</script>", "project_root": Path("/tmp/test")},
        {"project_name": "test", "project_root": Path("file:///etc/shadow")},
        {"project_name": "数据项目", "project_root": Path("/tmp/中文")},
    ]
    
    for i, project in enumerate(hostile_projects):
        success, result, error = hostile_test(
            create_project_config,
            f"hostile_project_{i}",
            **project
        )
        record_test(
            "create_project_config",
            "config.projects",
            "hostile_projects",
            success,
            error,
            severity="high" if "../../../" in str(project.get('project_name', '')) else "medium"
        )
    
    print(f"Completed {len([r for r in test_results['module'] if r == 'config.projects'])} project config tests")
    
finally:
    shutil.rmtree(test_dir, ignore_errors=True)

Completed 7 project config tests


## Section 3: Config System - Client Profiles (Corrected)

In [4]:
from siege_utilities import (
    create_client_profile, save_client_profile, load_client_profile,
    list_client_profiles, validate_client_profile
)

test_dir = Path(tempfile.mkdtemp())

try:
    # Hostile client profile inputs (WITH CORRECT SIGNATURES)
    # create_client_profile(client_name, client_code, contact_info, **kwargs)
    hostile_clients = [
        {"client_name": "'; DROP TABLE clients; --", "client_code": "SQL001",
         "contact_info": {"primary_contact": "Test", "email": "test@example.com"}},
        {"client_name": "<script>alert('xss')</script>", "client_code": "XSS001",
         "contact_info": {"primary_contact": "Test", "email": "evil@example.com"}},
        {"client_name": "../../../root", "client_code": "PATH001",
         "contact_info": {"primary_contact": "Test", "email": "test@example.com"}},
        {"client_name": "A" * 1000, "client_code": "LONG001",
         "contact_info": {"primary_contact": "Test", "email": "test@example.com"}},
        {"client_name": "test\x00client", "client_code": "NULL001",
         "contact_info": {"primary_contact": "Test", "email": "test@example.com"}},
        # Email injection attempts
        {"client_name": "test", "client_code": "EMAIL001",
         "contact_info": {"primary_contact": "Test", "email": "test@example.com\nBCC: attacker@evil.com"}},
        {"client_name": "test", "client_code": "EMAIL002",
         "contact_info": {"primary_contact": "Test", "email": "test@example.com\r\nTo: victim@target.com"}},
        # Unicode
        {"client_name": "中文客户", "client_code": "UNI001",
         "contact_info": {"primary_contact": "张三", "email": "test@example.com"}},
    ]
    
    for i, client in enumerate(hostile_clients):
        success, result, error = hostile_test(
            create_client_profile,
            f"hostile_client_{i}",
            **client
        )
        record_test(
            "create_client_profile",
            "config.clients",
            "hostile_clients",
            success,
            error,
            severity="critical" if "DROP TABLE" in str(client.get('client_name', '')) else "medium"
        )
        
        # Test validation if profile was created
        if success and result:
            success2, result2, error2 = hostile_test(
                validate_client_profile,
                f"validate_hostile_{i}",
                result
            )
            record_test(
                "validate_client_profile",
                "config.clients",
                "validate_hostile",
                success2,
                error2,
                severity="medium"
            )
    
    print(f"Completed {len([r for r in test_results['module'] if r == 'config.clients'])} client profile tests")
    
finally:
    shutil.rmtree(test_dir, ignore_errors=True)

[siege_utilities] 2025-10-13 08:37:35,100 INFO: Created client profile: '; DROP TABLE clients; -- (SQL001)


[siege_utilities] 2025-10-13 08:37:35,100 INFO: Client profile validation: 0 issues, 2 warnings


[siege_utilities] 2025-10-13 08:37:35,100 INFO: Created client profile: <script>alert('xss')</script> (XSS001)


[siege_utilities] 2025-10-13 08:37:35,101 INFO: Client profile validation: 0 issues, 2 warnings


[siege_utilities] 2025-10-13 08:37:35,101 INFO: Created client profile: ../../../root (PATH001)


...

[siege_utilities] 2025-10-13 08:37:35,102 INFO: Created client profile: test (EMAIL002)


[siege_utilities] 2025-10-13 08:37:35,102 INFO: Client profile validation: 0 issues, 2 warnings


[siege_utilities] 2025-10-13 08:37:35,102 INFO: Created client profile: 中文客户 (UNI001)


[siege_utilities] 2025-10-13 08:37:35,102 INFO: Client profile validation: 0 issues, 2 warnings


Completed 16 client profile tests


## Section 4: Git Utilities - Branch Analysis (Corrected)

In [5]:
from siege_utilities import analyze_branch_status, generate_branch_report

# Hostile repository paths (CORRECT SIGNATURES)
# analyze_branch_status(branch_name=None, repo_path=None)
# generate_branch_report(repo_path=None)

hostile_repo_paths = [
    Path("../../../etc/passwd"),
    Path("/nonexistent/repo"),
    Path("/dev/null"),
    Path("A" * 1000),
    Path("~/../../../../../../etc/shadow"),
]

for repo_path in hostile_repo_paths:
    # Test analyze_branch_status with hostile repo paths
    success, result, error = hostile_test(
        analyze_branch_status,
        "hostile_repo_path",
        repo_path=repo_path
    )
    record_test(
        "analyze_branch_status",
        "git.branch_analyzer",
        "hostile_repo_paths",
        success,
        error,
        severity="high" if "../../../" in str(repo_path) else "medium"
    )
    
    # Test generate_branch_report
    success2, result2, error2 = hostile_test(
        generate_branch_report,
        "hostile_repo_report",
        repo_path=repo_path
    )
    record_test(
        "generate_branch_report",
        "git.branch_analyzer",
        "hostile_repo_paths",
        success2,
        error2,
        severity="high" if "../../../" in str(repo_path) else "medium"
    )

# Test with hostile branch names
hostile_branch_names = [
    "'; DROP TABLE branches; --",
    "../../../etc/passwd",
    "branch; rm -rf /",
    "\x00null",
    "A" * 1000,
]

for branch_name in hostile_branch_names:
    success, result, error = hostile_test(
        analyze_branch_status,
        "hostile_branch_name",
        branch_name=branch_name,
        repo_path=None  # Use current directory
    )
    record_test(
        "analyze_branch_status",
        "git.branch_analyzer",
        "hostile_branch_names",
        success,
        error,
        severity="high"
    )

print(f"Completed {len([r for r in test_results['module'] if 'git' in r])} git utility tests")

Completed 15 git utility tests


## Section 5: Generate Corrected Phase 2 Results

In [6]:
# Convert results to DataFrame
df_phase2 = pd.DataFrame(test_results)

# Summary statistics
print("\n" + "="*80)
print("PHASE 2 CORRECTED HOSTILE TESTING SUMMARY")
print("="*80)
print(f"\nPhase 2 tests run: {len(df_phase2)}")
print(f"Phase 2 passed: {df_phase2['passed'].sum()}")
print(f"Phase 2 failed: {(~df_phase2['passed']).sum()}")
print(f"Phase 2 pass rate: {df_phase2['passed'].sum() / len(df_phase2) * 100:.1f}%")

# Functions tested
unique_functions = df_phase2['function'].nunique()
print(f"\nUnique functions tested: {unique_functions}")
print(f"Coverage: {unique_functions}/751 = {unique_functions/751*100:.1f}%")

# Summary by module
print("\n" + "="*80)
print("RESULTS BY MODULE")
print("="*80)
module_summary = df_phase2.groupby('module').agg({
    'passed': ['sum', 'count']
})
module_summary.columns = ['passed', 'total']
module_summary['pass_rate'] = (module_summary['passed'] / module_summary['total'] * 100).round(1)
print(module_summary)

# Critical failures
print("\n" + "="*80)
print("FAILURES BY SEVERITY")
print("="*80)
failures = df_phase2[~df_phase2['passed']]
if len(failures) > 0:
    severity_summary = failures.groupby('severity').size()
    print(severity_summary)
    
    print("\n" + "="*80)
    print("CRITICAL AND HIGH SEVERITY FAILURES")
    print("="*80)
    critical_high = failures[failures['severity'].isin(['critical', 'high'])]
    if len(critical_high) > 0:
        for idx, row in critical_high.iterrows():
            print(f"\nFunction: {row['function']}")
            print(f"Module: {row['module']}")
            print(f"Category: {row['test_category']}")
            print(f"Severity: {row['severity']}")
            print(f"Error: {row['error_message'][:200]}")
    else:
        print("No critical or high severity failures")
else:
    print("No failures detected")

# Save results
results_file = Path("hostile_testing_phase2_corrected_results.csv")
df_phase2.to_csv(results_file, index=False)
print(f"\nResults saved to: {results_file}")

# Display functions tested
print("\n" + "="*80)
print("FUNCTIONS TESTED")
print("="*80)
for func in df_phase2['function'].unique():
    func_tests = df_phase2[df_phase2['function'] == func]
    passed = func_tests['passed'].sum()
    total = len(func_tests)
    print(f"{func}: {passed}/{total} passed ({passed/total*100:.1f}%)")


PHASE 2 CORRECTED HOSTILE TESTING SUMMARY

Phase 2 tests run: 59
Phase 2 passed: 33
Phase 2 failed: 26
Phase 2 pass rate: 55.9%

Unique functions tested: 8
Coverage: 8/751 = 1.1%

RESULTS BY MODULE
                     passed  total  pass_rate
module                                       
config.clients           16     16      100.0
config.databases         17     21       81.0
config.projects           0      7        0.0
git.branch_analyzer       0     15        0.0

FAILURES BY SEVERITY
severity
high      10
medium    16
dtype: int64

CRITICAL AND HIGH SEVERITY FAILURES

Function: create_project_config
Module: config.projects
Category: hostile_projects
Severity: high
Error: create_project_config() missing 1 required positional argument: 'project_code'

Function: analyze_branch_status
Module: git.branch_analyzer
Category: hostile_repo_paths
Severity: high
Error: Not a git repository: /Users/dheerajchand/Desktop/etc/passwd

Function: generate_branch_report
Module: git.branch_analyze